# 03_03 Hiding the future

*📖 Book: section 3.5*

**How to use this notebook:** click a cell and press Shift+Enter to run just that cell.
Before each code cell, jot down what you think it will do. Then run it and see if you were right.

Run the cells in order, from the top. **A cell can use what earlier cells made**, so if you skip ahead you'll get `NameError: name '…' is not defined`. If that happens, run the cells above it, or just click Run All.

> **Where this fits**
>
> ```
> Odyssey text → numbers → [model reads] → model learns → model writes → MCP tool
> ```
>
> You are here: **model reads**
>
> Project files this connects to:
>
> - `src/homer_ai/model.py`
>
> **This notebook:** In practice the next letter is the answer, so a model that can see it would just copy it. The causal mask hides every later letter.
>
> For the big picture, what we're building and why, see the [course map](../00_start_here/00_00_course_map.ipynb).

## 1. What you'll learn

- Why a letter must not see the letters after it
- How -inf and softmax hide the future: softmax of -inf is 0
- How `torch.triu` builds the mask, and the triangle it makes
- Where the mask is in `model.py`, and a test that nothing leaks

## 2. Words you'll meet

| Word | Meaning in plain English | Example |
|---|---|---|
| **causal mask** | a table added to the scores that hides the letters after each one, so only earlier letters can influence it (cause before effect) | `torch.triu(..., diagonal=1)` |
| **-inf** | minus infinity: a number lower than every other number; softmax turns it into a share of exactly 0 | `float("-inf")` |

## 3. The idea

### ❓ Why can't a letter look at the letters after it?

Remember how the practice questions are made ([02_04](../02_text_to_numbers/02_04_sliding_window.ipynb)): the answer for each letter is simply the **next** letter. If the model could see the next letter, it would just copy it. That's a perfect score in practice, and useless when it has to write new text, where the future doesn't exist yet. It's like taking a test with the answers printed on the next page. No peeking!

For `ἄνδρα`, here's who may look at what:

```
the letter that guesses:   ἄ      ν      δ      ρ      α
may look at:               ἄ      ἄν     ἄνδ    ἄνδρ   ἄνδρα
must NOT see:              νδρα   δρα    ρα     α      (nothing)
```

(In the same way, when guessing the 5th letter of any text, letters 6 and later are hidden.) In the heatmap of [03_01](03_01_simple_attention.ipynb), row = the letter looking, column = the letter looked at. So the hidden letters are the cells **above the diagonal**: a triangle.

## 4. Step by step

Each step is one question. Read it, guess what the code will do, run it, then look at what you got.

### ❓ Step 1: how can softmax give a letter a share of exactly 0?

Inside, softmax turns each score `s` into `e^s` and then divides by the total. Here `e` is a fixed number, about 2.718. `e^s` is always positive, and bigger for a bigger `s` (`e^2` = 7.39, `e^1` = 2.72, `e^0` = 1). If a score is **minus infinity** (`-inf`, lower than any number), `e^-inf` is exactly 0, so that letter gets a share of exactly 0. So before softmax, we set the score of every hidden letter to `-inf`.

For `[2, 1, -inf]`: 7.39, 2.72 and 0 add up to 10.11, so the shares are 0.731, 0.269 and 0.

👉 **Predict first:** what will softmax give for the scores `[2.0, 1.0, -inf]`?

In [ ]:
import torch

scores = torch.tensor([2.0, 1.0, float("-inf")])
print(torch.softmax(scores, dim=0))
print(torch.softmax(scores[:2], dim=0), "<- the same, with the third score left out")

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

`[0.7311, 0.2689, 0.0000]`. The hidden letter gets exactly 0, and the other two share all of the 1. It's just as if the third score had been left out.

</details>

### ❓ Step 2: how do we make a table with `-inf` above the diagonal?

With `torch.triu` ("**tri**angle **u**pper"). It keeps the upper triangle of a table and sets everything else to 0. `diagonal=1` means "start one step above the main diagonal", so each letter still sees itself. We build it in small steps for 5 letters.

👉 **Predict first:** after `triu`, which boxes will be `-inf` and which 0?

In [ ]:
ones = torch.ones(5, 5)
print(ones * float("-inf"))  # a table full of -inf
mask = torch.triu(ones * float("-inf"), diagonal=1)
print(mask)

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

The first table is `-inf` everywhere. After `triu(..., diagonal=1)` only the boxes **above** the diagonal are `-inf`, and the diagonal and everything below are `0`. Adding 0 to a score changes nothing, and adding `-inf` hides it. So `scores + mask` hides exactly the future.

</details>

### ❓ Your turn: can you build the mask for 3 letters?

The mask above is for 5 letters. Build the one for 3, then see what it does to three letters whose scores are all 0.

👉 **Predict first:** with all scores equal, what shares does the third letter give to letters 1, 2 and 3? And the first letter?

Replace each `None`, run the cell, then run the check cell under it.

In [ ]:
# 👉 do: a 3×3 mask: -inf above the diagonal, 0 on it and below it
small_mask = None

# 👉 do: add the mask to 3×3 scores that are all 0, then softmax along each row
small_shares = None

In [ ]:
from homer_ai.lessons import check

check(
    "the 3×3 mask",
    small_mask,
    [[0.0, float("-inf"), float("-inf")], [0.0, 0.0, float("-inf")], [0.0, 0.0, 0.0]],
    coach="If the diagonal is -inf too, you left out diagonal=1: then the first "
    "letter can't even see itself, and softmax of a row of only -inf gives nan. "
    "If the -inf are BELOW the diagonal, you used tril: that hides the past.",
)
check(
    "the shares of 3 letters with equal scores",
    small_shares,
    [[1.0, 0.0, 0.0], [0.5, 0.5, 0.0], [1 / 3, 1 / 3, 1 / 3]],
    coach="Each ROW has to add up to 1, so softmax runs along the rows: dim=-1. "
    "And the mask is added to the scores before softmax, not after.",
)

<details><summary>💡 Hint</summary>

The mask is the line of step 2, `torch.triu(ones * float("-inf"), diagonal=1)`, with a 3 × 3 table of ones. The scores are `torch.zeros(3, 3)`.

</details>

<details><summary>✅ Answer</summary>

```python
small_mask = torch.triu(torch.ones(3, 3) * float("-inf"), diagonal=1)
small_shares = torch.softmax(torch.zeros(3, 3) + small_mask, dim=-1)
print(small_shares)
```

The shares come out as a staircase:

```
1.00  0.00  0.00
0.50  0.50  0.00
0.33  0.33  0.33
```

The first letter sees only itself, the second shares between two, the third between three. Nothing goes to a later letter.

</details>

### ❓ Step 3: what does it look like on the real attention of `ἄνδρα`?

We use our model's real queries, keys and values (as in [03_02](03_02_trainable_attention.ipynb)), written once as a function that can switch the mask on or off. Then we draw the weights as a heatmap.

👉 **Predict first:** what will the top row (the letter `ἄ`, which may only see itself) be?

In [ ]:
import torch

from homer_ai.lessons import load_homer_model

# read only: we load the saved model, we never save
tokenizer, model = load_homer_model(fallback_untrained=True)

In [ ]:
import matplotlib.pyplot as plt

from homer_ai.lessons import draw_weights  # the heatmap function, drawn in 03_01
from homer_ai.studio_style import studio_figure

In [ ]:
def attend(x, masked):
    Q, K, V = model.query(x), model.key(x), model.value(x)
    scores = (Q @ K.T) / (x.shape[-1] ** 0.5)
    if masked:
        n = len(x)
        scores = scores + torch.triu(torch.ones(n, n) * float("-inf"), diagonal=1)
    weights = torch.softmax(scores, dim=-1)
    return weights, weights @ V


ids = torch.tensor(tokenizer.encode("ἄνδρα"))
with torch.no_grad():
    weights, context = attend(model.embedding(ids), masked=True)

fig = studio_figure("The causal mask", "The future is hidden: a triangle", (8, 6.5))
draw_weights(
    fig, (0.14, 0.08, 0.66, 0.62), weights.numpy(), list("ἄνδρα"),
    "attention weights with the mask: each row adds up to 1",
)
plt.show()
print("each row adds up to:", weights.sum(dim=-1))

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

A triangle. Everything above the diagonal is exactly `0.00`. The top row `ἄ` is `[1, 0, 0, 0, 0]` (it can only see itself), the second row spreads its share over `ἄ` and `ν`, and so on. Every row still adds up to 1, because softmax shares out the 1 among the letters that are left.

</details>

### ❓ Can I prove nothing leaks?

Here's a strong test: change only the **last** letter (`ἄνδρα` → `ἄνδρο`) and see whether the output for the earlier letters changes. With the mask it mustn't, because the earlier letters can't see the last one. We test the mixed result of each letter, with the mask on and off, and then the real model's final scores for the first four positions.

👉 **Predict first:** with the mask on, which of the 5 letters change? And with the mask off?

In [ ]:
other = torch.tensor(tokenizer.encode("ἄνδρο"))  # only the last letter differs

with torch.no_grad():
    for masked in (True, False):
        _, context_a = attend(model.embedding(ids), masked)
        _, context_b = attend(model.embedding(other), masked)
        unchanged = [torch.allclose(context_a[i], context_b[i]) for i in range(5)]
        state = "on " if masked else "off"
        print(f"mask {state}: context unchanged for ἄ ν δ ρ α? {unchanged}")

    same_real = torch.allclose(model(ids)[:4], model(other)[:4])
    print("real model: first 4 outputs unchanged?", same_real)

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

With the mask **on**, the first four letters are unchanged (`True`) and only the last changes, because the last letter itself changed. With the mask off, all five change. The earlier letters had been peeking at the last one. And the real model passes the test too.

</details>

### ❓ Where is the mask in `model.py`?

Here are the real lines, printed straight from the code. Our function above does the same thing and, as the check below shows, gives the same answer as the real model.

👉 **Predict first:** which two lines mention the mask?

In [ ]:
import inspect

from homer_ai.model import HomerGreekLLM

for line in inspect.getsource(HomerGreekLLM.forward).splitlines():
    if "mask" in line or "softmax" in line:
        print(line.strip())

with torch.no_grad():
    mine = model.head(attend(model.embedding(ids), masked=True)[1])
    print("same scores as the real model?", torch.allclose(mine, model(ids)))

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

`mask = torch.triu(torch.ones(seq_len, seq_len) * float("-inf"), diagonal=1)` builds the triangle, and `weights = F.softmax(scores + mask, dim=-1)` adds it to the scores before softmax. The check prints `True`, so our function and the real model agree.

</details>

### ❓ Can the model still tell the order of the letters before the one it is guessing from?

We can finally test what [02_06](../02_text_to_numbers/02_06_positions.ipynb) promised. We give our real `HomerGreekLLM` two inputs with the same last letter, where the letters before it are only swapped: `ἄνδρα` and `νἄδρα` (the first two letters swapped).

👉 **Predict first:** will its scores after the final `α` (137 numbers) be the same for both?

In [ ]:
first = torch.tensor(tokenizer.encode("ἄνδρα"))
swapped = torch.tensor(tokenizer.encode("νἄδρα"))

with torch.no_grad():
    scores_first, scores_swapped = model(first), model(swapped)
same_first = torch.allclose(scores_first[0], scores_swapped[0])
same_last = torch.allclose(scores_first[-1], scores_swapped[-1], atol=1e-5)
print("after the 1st letter, same scores?", same_first)
print("after the last letter, same scores?", same_last)

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

After the first letter the scores differ, because the first letters themselves differ (`ἄ` against `ν`). After the last letter they're **identical**. The model can't tell the order of the letters before it, because attention mixes their rows with no idea where each one sits. That's what the position table from chapter 2 is for, and `model.py` doesn't have one.

</details>

### ❓ What is dropout, and does our model use it?

**Dropout** is a training trick. At random, it switches off some of the numbers (the attention weights, for example), so the model can't lean too much on any single one. Our `model.py` doesn't use dropout.

## 5. Recap

- A letter must not see later letters, or it would copy the answer; they are the cells above the diagonal.
- We add `-inf` to those scores before softmax: softmax of `-inf` is exactly 0, and the other shares still add up to 1.
- `torch.triu(ones * -inf, diagonal=1)` builds the mask, and it is one line of `model.py`; a changed last letter does not move the earlier outputs.

**You can now:**

- build the causal mask for any number of letters with `torch.triu`
- say why the hidden scores are `-inf` and not 0
- prove nothing leaks: change the last letter and compare the earlier outputs

**Next:** [03_04: Many heads](03_04_many_heads.ipynb): what if several attention heads look at once?